# 使用 Realtime API 构建数据密集型应用的实用指南

本 cookbook 是一份实用指南，旨在帮助 AI 工程师最大限度地发挥 OpenAI Realtime API 的效能，尤其是在处理数据密集型函数调用时。我们将重点关注语音到语音智能体中的常见场景，在这些场景中，大量数据必须得到顺畅、高效的处理。

本文不会介绍搭建 Realtime API 方案的基础知识。相反，你将获得清晰的见解和可落地的策略，用于提升实时对话智能体的性能和可靠性。本文针对的是在实时对话场景中处理大量数据时特有的挑战。

### 什么是 Realtime API？

在深入之前，先为初次接触的读者快速回顾一下这个 API。OpenAI Realtime API 是一项较新的服务，支持低延迟的多模态交互——例如语音到语音的对话和实时转录。可以想象基于语音的实时客服或电影实时转录等场景。 

### 什么是数据密集型函数调用？

智能体需要访问工具和相关数据才能完成任务。例如，金融分析师智能体可能需要拉取实时市场数据。在很多情况下，你的环境中已经存在通过 API 公开这些信息的服务。

过去，API 在设计时并没有考虑智能体，根据服务的不同，它们往往会返回大量数据。作为工程师，我们经常用函数调用来封装这些 API，以加快智能体的开发——这完全合理。既然已经有现成的，何必重复造轮子？

如果不经过仔细优化，这些数据密集型函数调用很快就会让 Realtime API 不堪重负——导致响应缓慢，甚至无法处理用户请求。

### 场景设定

我们的示例围绕一个 NBA 球探智能体展开，它会调用多个函数，对即将参加选秀的新秀进行深入分析。为了演示 Realtime API 交互的实用准则，我们使用了受 NBA 选秀新秀启发、规模庞大且贴近真实的负载。下面是在 Realtime 会话中定义的一个大而全的 `searchDraftProspects` 函数，用于设定场景。

```json
// "Hey, pull up point guards projected in the top 10 in the 2025 draft"
{
  "type": "session.update",
  "session": {
    "tools": [
      {
        "type": "function",
        "name": "searchDraftProspects",
        "description": "Search draft prospects for a given year e.g., Point Guard",
        "parameters": {
          "type": "object",
          "properties": {
            "sign": {
              "type": "string",
              "description": "The player position",
              "enum": [
                "Point Guard",
                "Shooting Guard",
                "Small Forward",
                "Power Forward",
                "Center",
                "Any"
              ]
            },
            year: { type: "number", description: "Draft year e.g., 2025" },
            mockDraftRanking: { type: "number", description: "Predicted Draft Ranking" },
          },
          "required": ["position", "year"]
        }
      }
    ],
    "tool_choice": "auto",
  }
}
```

searchDraftProspects 函数调用会返回一个庞大的负载。该示例的结构和规模取自我们遇到过的真实场景。

```json
// Example Payload
{
  "status": {
    "code": 200,
    "message": "SUCCESS"
  },
  "found": 4274,
  "offset": 0,
  "limit": 10,
  "data": [
    {
      "prospectId": 10001,
      "data": {
        "ProspectInfo": {
          "league": "NCAA",
          "collegeId": 301,
          "isDraftEligible": true,
          "Player": {
            "personalDetails": {
              "firstName": "Jalen",
              "lastName": "Storm",
              "dateOfBirth": "2003-01-15",
              "nationality": "USA"
            },
            "physicalAttributes": {
              "position": "PG",
              "height": {
                "feet": 6,
                "inches": 4
              },
              "weightPounds": 205
            },
            "hometown": {
              "city": "Springfield",
              "state": "IL"
            }
          },
          "TeamInfo": {
            "collegeTeam": "Springfield Tigers",
            "conference": "Big West",
            "teamRanking": 12,
            "coach": {
              "coachId": 987,
              "coachName": "Marcus Reed",
              "experienceYears": 10
            }
          }
        },
        "Stats": {
          "season": "2025",
          "gamesPlayed": 32,
          "minutesPerGame": 34.5,
          "shooting": {
            "FieldGoalPercentage": 47.2,
            "ThreePointPercentage": 39.1,
            "FreeThrowPercentage": 85.6
          },
          "averages": {
            "points": 21.3,
            "rebounds": 4.1,
            "assists": 6.8,
            "steals": 1.7,
            "blocks": 0.3
          }
        },
        "Scouting": {
          "evaluations": {
            "strengths": ["Court vision", "Clutch shooting"],
            "areasForImprovement": ["Defensive consistency"]
          },
          "scouts": [
            {
              "scoutId": 501,
              "name": "Greg Hamilton",
              "organization": "National Scouting Bureau"
            }
          ]
        },
        "DraftProjection": {
          "mockDraftRanking": 5,
          "lotteryPickProbability": 88,
          "historicalComparisons": [
            {
              "player": "Chris Paul",
              "similarityPercentage": 85
            }
          ]
        },
        "Media": {
          "highlightReelUrl": "https://example.com/highlights/jalen-storm",
          "socialMedia": {
            "twitter": "@jstorm23",
            "instagram": "@jstorm23_ig"
          }
        },
        "Agent": {
          "agentName": "Rick Allen",
          "agency": "Elite Sports Management",
          "contact": {
            "email": "rallen@elitesports.com",
            "phone": "555-123-4567"
          }
        }
      }
    },
    // ... Many thousands of tokens later.
  ]
}
```

## 指导原则

### 1. 将臃肿的函数拆分为职责明确的更小函数

这几乎不言而喻——构建函数调用时，首要任务是设计清晰、定义明确的函数。这样可以轻松缩减响应大小，避免让模型不堪重负。每个函数调用都应当易于解释、范围明确，并且只返回实现其目的所需的信息。函数之间职责重叠必然会导致混乱。

例如，我们可以限制 `searchDraftProspects` 函数调用只返回每位新秀的一般信息（例如球员数据），从而大幅缩减响应大小。如果需要更多信息，可以通过新的 `getProspectDetails` 函数调用获取更详细的内容。没有放之四海而皆准的方案；合适的做法取决于你的用例和数据模型。

```json
{
  "tools": [
    {
      "type": "function",
      "name": "searchDraftProspects",
      "description": "Search NBA draft prospects by position, draft year, and projected ranking, returning only general statistics to optimize response size.",
      "parameters": {
        "type": "object",
        "properties": {
          "position": {
            "type": "string",
            "description": "The player's basketball position.",
            "enum": [
              "Point Guard",
              "Shooting Guard",
              "Small Forward",
              "Power Forward",
              "Center",
              "Any"
            ]
          },
          "year": {
            "type": "number",
            "description": "Draft year, e.g., 2025"
          },
          "maxMockDraftRanking": {
            "type": "number",
            "description": "Maximum predicted draft ranking (e.g., top 10)"
          }
        },
        "required": ["position", "year"]
      }
    },
    {
      "type": "function",
      "name": "getProspectDetails",
      "description": "Fetch detailed information for a specific NBA prospect, including comprehensive stats, agent details, and scouting reports.",
      "parameters": {
        "type": "object",
        "properties": {
          "playerName": {
            "type": "string",
            "description": "Full name of the prospect (e.g., Jalen Storm)"
          },
          "year": {
            "type": "number",
            "description": "Draft year, e.g., 2025"
          },
          "includeAgentInfo": {
            "type": "boolean",
            "description": "Include agent information"
          },
          "includeStats": {
            "type": "boolean",
            "description": "Include detailed player statistics"
          },
          "includeScoutingReport": {
            "type": "boolean",
            "description": "Include scouting report details"
          }
        },
        "required": ["playerName", "year"]
      }
    }
  ],
  "tool_choice": "auto"
}
```

### 2. 随着对话推进，优化上下文

Realtime 对话允许长达 30 分钟的会话——但滚动上下文窗口仅支持约 16,000 个 token（取决于模型快照，上下文窗口的限制正在不断改善）。因此，在较长的交流过程中，你可能会注意到性能逐渐下降。随着对话推进、函数调用增多，对话状态会迅速膨胀，其中既有重要信息，也有不必要的噪声——因此重点保留最相关的细节非常重要。这种做法有助于保持良好的性能并降低成本。

**i) 定期总结对话状态**

在对话推进过程中定期进行总结，是缩减上下文规模的绝佳方式——既能降低成本，也能降低延迟。

请参阅 @Minhajul 关于在 Realtime 对话中实现自动总结的精彩指南（[链接](https://cookbook.openai.com/examples/context_summarization_with_realtime_api)）。

**ii) 定期提醒模型其角色和职责**

数据量大的负载会很快填满上下文窗口。如果你注意到模型开始忘记指令或可用工具，可以通过调用 `session.update` 定期提醒它系统提示词和工具——这能让它始终专注于自己的角色和职责。

### 3. 数据处理与优化

**i) 在函数调用中使用过滤，将数据量大的响应精简到只包含回答问题所需的关键字段**

一般来说，函数调用返回的 token 越少，响应质量就越好。常见的陷阱是函数调用返回了长达数千 token 的超大负载。请着重在每个函数调用中应用过滤（无论是在数据层面还是函数层面），以尽量缩减响应大小。

```json
// Filtered response
{
  "status": {
    "code": 200,
    "message": "SUCCESS"
  },
  "found": 4274,
  "offset": 0,
  "limit": 5,
  "data": [
    {
    "zpid": 7972122,
      "data": {
        "PropertyInfo": {
            "houseNumber": "19661",
            "directionPrefix": "N ",
            "streetName": "Central",
            "streetSuffix": "Ave",
            "city": "Phoenix",
            "state": "AZ",
            "postalCode": "85024",
            "zipPlusFour": "1641"
            "bedroomCount": 2,
            "bathroomCount": 2,
            "storyCount": 1,
            "livingAreaSize": 1089,
            "livingAreaSizeUnits": "Square Feet",
            "yearBuilt": "1985"
          }
		    }
			}
		]
		// ... 
}
```

**ii) 扁平化层级负载——同时不丢失关键信息**

API 调用返回的层级负载有时会包含重复的层级标题——例如 "ProspectInfo" 或 "Stats"——这可能会增加额外的噪声，让模型更难处理。在探索如何让数据更高效时，你可以尝试裁剪掉一些不必要的标签，将这些结构扁平化。这有助于提升性能，但请考虑在你的特定用例中哪些信息是需要保留的。

```json
// Flattened payload
{
  "status": {
    "code": 200,
    "message": "SUCCESS"
  },
  "found": 4274,
  "offset": 0,
  "limit": 2,
  "data": [
    {
      "prospectId": 10001,
      "league": "NCAA",
      "collegeId": 301,
      "isDraftEligible": true,
      "firstName": "Jalen",
      "lastName": "Storm",
      "position": "PG",
      "heightFeet": 6,
      "heightInches": 4,
      "weightPounds": 205,
      "hometown": "Springfield",
      "state": "IL",
      "collegeTeam": "Springfield Tigers",
      "conference": "Big West",
      "teamRanking": 12,
      "coachId": 987,
      "coachName": "Marcus Reed",
      "gamesPlayed": 32,
      "minutesPerGame": 34.5,
      "FieldGoalPercentage": 47.2,
      "ThreePointPercentage": 39.1,
      "FreeThrowPercentage": 85.6,
      "averagePoints": 21.3,
      "averageRebounds": 4.1,
      "averageAssists": 6.8,
      "stealsPerGame": 1.7,
      "blocksPerGame": 0.3,
      "strengths": ["Court vision", "Clutch shooting"],
      "areasForImprovement": ["Defensive consistency"],
      "mockDraftRanking": 5,
      "lotteryPickProbability": 88,
      "highlightReelUrl": "https://example.com/highlights/jalen-storm",
      "agentName": "Rick Allen",
      "agency": "Elite Sports Management",
      "contactEmail": "rallen@elitesports.com"
    },
		...
 }
 ```

**iii) 尝试不同的数据格式**

数据的组织方式会直接影响模型处理和总结 API 响应的效果。根据我们的经验，与 Markdown 等表格格式相比，JSON 或 YAML 这类清晰的、基于键的格式能帮助模型更准确地解读数据。尤其是大型表格，往往会让模型不堪重负——导致输出不够流畅、准确性下降。不过，仍然值得尝试不同的格式，找到最适合你用例的方式。

```yaml
status:
  code: 200
  message: "SUCCESS"
found: 4274
offset: 0
limit: 10
data:
  - prospectId: 10001
    data:
      ProspectInfo:
        league: "NCAA"
        collegeId: 301
        isDraftEligible: true
        Player:
          firstName: "Jalen"
          lastName: "Storm"
          position: "PG"
          heightFeet: 6
          heightInches: 4
          weightPounds: 205
          hometown: "Springfield"
          state: "IL"
        TeamInfo:
          collegeTeam: "Springfield Tigers"
          conference: "Big West"
          teamRanking: 12
          coachId: 987
          coachName: "Marcus Reed"
      Stats:
        gamesPlayed: 32
        minutesPerGame: 34.5
        FieldGoalPercentage: 47.2
        ThreePointPercentage: 39.1
        FreeThrowPercentage: 85.6
        averagePoints: 21.3
        averageRebounds: 4.1
        averageAssists: 6.8
        stealsPerGame: 1.7
        blocksPerGame: 0.3
      Scouting:
        strengths:
          - "Court vision"
          - "Clutch shooting"
        areasForImprovement:
          - "Defensive consistency"
      DraftProjection:
        mockDraftRanking: 5
        lotteryPickProbability: 88
      Media:
        highlightReelUrl: "https://example.com/highlights/jalen-storm"
      Agent:
        agentName: "Rick Allen"
        agency: "Elite Sports Management"
        contactEmail: "rallen@elitesports.com"
```

## 4. 在数据量大的函数调用之后，跟进提示性提示词

底层模型往往难以从数据量大的响应顺畅地过渡到准确的回答。为了在处理复杂数据时提升流畅性和准确性，请在函数调用之后立即提供一个函数调用提示（hint）。这些提示会就具体任务引导模型——教它如何解读关键字段和特定领域的取值。

下面的示例展示了一个有效的提示性提示词。

```javascript
// Function call hint
let prospectSearchPrompt = `
Parse NBA prospect data and provide a concise, engaging response.

General Guidelines
- Act as an NBA scouting expert.
- Highlight key strengths and notable attributes.
- Use conversational language.
- Mention identical attributes once.
- Ignore IDs and URLs.

Player Details
- State height conversationally ("six-foot-eight").
- Round weights to nearest 5 lbs.

Stats & Draft Info
- Round stats to nearest whole number.
- Use general terms for draft ranking ("top-five pick").
Experience
- Refer to players as freshman, sophomore, etc., or mention professional experience.
- Location & TeamMention hometown city and state/country.
- Describe teams conversationally.

Skip (unless asked explicitly)
- Exact birth dates
- IDs
- Agent/contact details
- URLs

Examples
- "Jalen Storm, a dynamic six-foot-four point guard from Springfield, Illinois, averages 21 points per game."
- "Known for his clutch shooting, he's projected as a top-five pick."

Important: Respond based strictly on provided data, without inventing details.
`;
```

在实践中，我们先将函数调用结果追加到对话中，然后通过 Realtime API 使用提示性提示词生成一个响应。瞧——模型就能从容地处理所有信息了。

```javascript
// Add new conversation item for the model
const conversationItem = {
  type: 'conversation.item.create',
  previous_item_id: output.id,
  item: {
    call_id: output.call_id,
    type: 'function_call_output',
    output: `Draft Prospect Search Results: ${result}`
  }
};

dataChannel.send(JSON.stringify(conversationItem));

// Emit a response from the model including the hint prompt
const event = {
  type: 'response.create',
  conversation: "none",
  response: {
    instructions: prospectSearchPrompt # function call hint
  }
};

dataChannel.send(JSON.stringify(event));
```

## 总结

使用 Realtime API 构建高效的智能体，是一个不断探索和调整的过程。

**关键建议汇总**

- **过滤数据：** 只包含与用户请求或模型下一步直接相关的字段和细节，其余的全部裁剪掉。
- **扁平化并简化结构：** 减少深层嵌套或冗余的数据。以便于模型和人类快速浏览的方式呈现信息。
- **优先使用清晰的结构化格式：** 使用字段名一致、噪声最少的 JSON（或 YAML）。对于数据量大的响应，避免使用大型表格或 markdown。
- **用提示性提示词引导模型：** 在返回大量数据后，紧接着提供一个有针对性的提示词，明确说明模型应当提取或总结哪些内容。

请记住——实验至关重要。Realtime 模型在不断改进，我们也会持续分享技巧，帮助你充分发挥 Realtime API 的作用。